## Search with Nepali Query on available files inside the data directory

In [1]:
import joblib
import numpy as np
import scipy.sparse as sp
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nepali_pipeline
from nepali_pipeline import identity_tokenizer

DATA_DIR = Path("data")
NEPALI_LEMMA_PIPELINE_PATH = Path("nepali_hmm_pipeline.pkl")
TFIDF_VECTORIZER_PATH = Path("tfidf_vectorizer.pkl")
TFIDF_MATRIX_PATH = Path("tfidf_matrix.npz")
FILE_INDEX_PATH = Path("tfidf_file_index.joblib")

## Create a pipeline for TF-IDF vectorization and load the trained model

### 1. Scan the `data/` directory and load the corpus

Every `.txt` file under `data/` is one document. A handful of the files have a single
stray non-UTF-8 byte in them (legacy scraping artifact), so we decode with
`errors="replace"` rather than letting the whole pipeline crash on 13 files out of 1291.

In [2]:
file_paths = sorted(DATA_DIR.glob("*.txt"))

documents = []
for path in file_paths:
    with open(path, encoding="utf-8-sig", errors="replace") as fh:
        documents.append(fh.read())

print(f"Loaded {len(documents)} documents from '{DATA_DIR}/'")
print("Example file:", file_paths[0].name)
print(documents[0][:200])

Loaded 1291 documents from 'data/'
Example file: 1405693439.txt

﻿प्रश्न  मधेशी र जनजातिदलहरुसंग यहाँहरुले मोर्चा गठन गर्नु भएको छ । पहिलो संविधान सभा पनि यस्तै मोर्चा र ध्रुबीकरणले विफल भएको निष्कर्ष छ । कतै यो संविधान सभा पनि विफल हुने संकेतहरुत होईन ?
उत्तर  यो


### 2. Load the Nepali lemmatizer pipeline defined in `nepali_pipeline.py`

`nepali_hmm_pipeline.pkl` is a fitted `sklearn.Pipeline` built entirely out of the classes in
[nepali_pipeline.py](nepali_pipeline.py): `NepaliLexicalAnalyzer` (tokenizer) &rarr;
`NepaliHMMLemmatizer` (the trained HMM) &rarr; `NepaliStopWordRemover`. We load it and call
`.transform()` on it directly -- not a generic scikit-learn `FunctionTransformer` wrapper -- so every
document is lemmatized by that actual custom pipeline before TF-IDF ever sees it.

Each document's lemmas are then rejoined into one whitespace-separated string, and `TfidfVectorizer`
is given `identity_tokenizer` (also from `nepali_pipeline.py`) as its tokenizer: it re-splits that
string with the same rule `NepaliLexicalAnalyzer` uses, so the vectorizer's vocabulary is built from
the exact lemmas produced above rather than from raw, uninflected words.

In [3]:
nepali_lemma_pipeline = joblib.load(NEPALI_LEMMA_PIPELINE_PATH)
print(nepali_lemma_pipeline)

lemmatized_documents = nepali_lemma_pipeline.transform(documents)
joined_documents = [" ".join(lemmas) for lemmas in lemmatized_documents]
print("\nExample lemmas:", lemmatized_documents[0][:15])

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=identity_tokenizer,
    token_pattern=None,
    lowercase=False,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(joined_documents)
print("\nTF-IDF matrix shape (documents x vocabulary):", tfidf_matrix.shape)

Pipeline(steps=[('tokenizer', NepaliLexicalAnalyzer()),
                ('lemmatizer', NepaliHMMLemmatizer()),
                ('stop_word_remover',
                 NepaliStopWordRemover(stop_words={'अक्सर', 'अघि', 'अझै', 'अतः',
                                                   'अति', 'अथवा', 'अधिक', 'अनि',
                                                   'अन्य', 'अन्यत्र', 'अन्यथा',
                                                   'अब', 'अरु', 'अर्को',
                                                   'अर्थात्', 'अलग', 'आज',
                                                   'आदि', 'आफू', 'आफ्नो',
                                                   'इत्यादि', 'उ', 'उनको',
                                                   'उनलाई', 'उनले', 'उनी',
                                                   'उनीहरु', 'उनीहरुको',
                                                   'उनीहरुलाई', 'उसको', ...}))])

Example lemmas: ['\ufeffप्रश्न', 'मधेशी', 'जनजातिदलहरुसंग', 'मोर्चा', 'गठन', 'गर्न

### 3. Write the matrix and the fitted model to disk

The lemmatizer pipeline is already persisted (`nepali_hmm_pipeline.pkl`), so only the newly-fitted
`tfidf_vectorizer` needs saving here, alongside the TF-IDF matrix as a sparse `.npz` file and the
row-index &rarr; file-path mapping needed to translate a matrix row back into a document.

In [4]:
joblib.dump(tfidf_vectorizer, TFIDF_VECTORIZER_PATH)
sp.save_npz(TFIDF_MATRIX_PATH, tfidf_matrix)
joblib.dump([str(p) for p in file_paths], FILE_INDEX_PATH)

print(f"Saved vectorizer -> {TFIDF_VECTORIZER_PATH}")
print(f"Saved matrix     -> {TFIDF_MATRIX_PATH}  ({tfidf_matrix.shape[0]} docs x {tfidf_matrix.shape[1]} terms)")
print(f"Saved file index -> {FILE_INDEX_PATH}")

Saved vectorizer -> tfidf_vectorizer.pkl
Saved matrix     -> tfidf_matrix.npz  (1291 docs x 43980 terms)
Saved file index -> tfidf_file_index.joblib


### 4. Search: rank files against a user query by TF-IDF cosine similarity

A query is lemmatized with the same `nepali_lemma_pipeline` and vectorized with the same fitted
`tfidf_vectorizer` (`.transform()`, not `.fit_transform()`, so it lands in the existing vector space
rather than starting a new one), then ranked against every document by cosine similarity -- the
standard vector-space-model retrieval method.

In [5]:
def search(query, lemma_pipeline, vectorizer, matrix, paths, top_n=5):
    """Rank every document in `paths` against `query` by TF-IDF cosine similarity."""
    query_lemmas = lemma_pipeline.transform([query])[0]
    query_vector = vectorizer.transform([" ".join(query_lemmas)])
    scores = cosine_similarity(query_vector, matrix).ravel()
    ranked = np.argsort(scores)[::-1][:top_n]
    return [(paths[i], float(scores[i])) for i in ranked if scores[i] > 0]


results = search("सेयर बजार घट्यो", nepali_lemma_pipeline, tfidf_vectorizer, tfidf_matrix, file_paths)
for path, score in results:
    print(f"{score:.4f}  {path.name}")

0.3773  1467213780.txt
0.3718  1466760240.txt
0.3672  1460566620.txt
0.3159  1462517220.txt
0.3157  1457603280.txt


### 5. Confirm the persisted model works on its own (fresh load, no in-memory state)

This reloads the lemmatizer pipeline, vectorizer, matrix, and file index purely from disk --
simulating a new session that only has `nepali_hmm_pipeline.pkl`, `tfidf_vectorizer.pkl`,
`tfidf_matrix.npz`, and `tfidf_file_index.joblib` -- and prints the winning file's own text as a
sanity check that the returned file actually matches the query.

In [6]:
loaded_lemma_pipeline = joblib.load(NEPALI_LEMMA_PIPELINE_PATH)
loaded_vectorizer = joblib.load(TFIDF_VECTORIZER_PATH)
loaded_matrix = sp.load_npz(TFIDF_MATRIX_PATH)
loaded_paths = [Path(p) for p in joblib.load(FILE_INDEX_PATH)]

query = "नेप्से परिसूचक घट्यो"
top_path, top_score = search(query, loaded_lemma_pipeline, loaded_vectorizer, loaded_matrix, loaded_paths, top_n=1)[0]

print(f"Query: {query!r}")
print(f"Best match: {top_path.name}  (score={top_score:.4f})\n")
print(top_path.read_text(encoding="utf-8-sig", errors="replace")[:300])

Query: 'नेप्से परिसूचक घट्यो'
Best match: 1451447700.txt  (score=0.3593)


﻿नयाँ पत्रिका काठमाडौं, १४ पुस
मंगलबार सेयर बजार परिसूचक नेप्से १० दशमलव २२ अंकले बढेको छ । अघिल्लो दिन सोमबार ३ दशमलव ५२ अंकले घटेर १ हजार १ सय ३१ दशमलव ९५ बिन्दुमा ओर्लिएको नेप्से मंगलबार बढेर १ हजार १ सय ४२ दशमलव १७ बिन्दुमा पुगेर रोकियो । आइतबार र सोमबार नेप्सेमा गिरावट आएको थियो ।
मंगलबार कुल 


In [4]:
import math
import numpy as np
import pandas as pd
#  documents and query
doc1 = (
    "Shadows stretched long across the quiet cobblestone alley as the evening"
    " train whistled in the distance."
)
doc2 = (
    "Quantum computers leverage the principles of superposition to process"
    " complex datasets at unprecedented speeds."
)
doc3 = (
    "A single ripe mango dropped silently onto the soft damp grass beneath"
    " the orchard canopy."
)
query = "The mango fell from the tree."

documents = [doc1, doc2, doc3]


# Simple helper function to clean and split text into lowercase words
def tokenize(text):
    text = text.lower().replace(".", "")  # remove simple punctuation
    return text.split()


# Tokenize all texts
doc_tokens = [tokenize(doc) for doc in documents]
query_tokens = tokenize(query)

#  Build the Vocabulary (unique words across all documents)
vocab = sorted(list(set(word for doc in doc_tokens for word in doc)))


# TF-IDF Functions
def compute_tf(tokens, vocab):
    """TF = (Count of word in doc) / (Total words in doc)"""
    doc_len = len(tokens)
    return [tokens.count(word) / doc_len for word in vocab]


def compute_idf(doc_tokens_list, vocab):
    """IDF = log10(Total documents / Documents containing the word)"""
    N = len(doc_tokens_list)
    idf_list = []
    for word in vocab:
        docs_containing_word = sum(
            1 for doc in doc_tokens_list if word in doc
        )
        # Adding 1 to avoid division by zero if a query word isn't in any doc
        idf_val = math.log10(N / (docs_containing_word or 1))
        idf_list.append(idf_val)
    return idf_list


# Compute Document TF-IDF Matrix
tf_matrix = [compute_tf(doc, vocab) for doc in doc_tokens]
idf_weights = compute_idf(doc_tokens, vocab)

# TF-IDF = TF * IDF
tfidf_matrix = np.array(tf_matrix) * np.array(idf_weights)

# Compute Query TF-IDF Vector (using the same IDF weights)
query_tf = compute_tf(query_tokens, vocab)
query_tfidf = np.array(query_tf) * np.array(idf_weights)

# TF-IDF Matrix in Pandas
df_tfidf = pd.DataFrame(
    tfidf_matrix, index=["doc1", "doc2", "doc3"], columns=vocab
)



#  Cosine Similarity Function
def cosine_similarity(vec1, vec2):
    """Cosine Similarity = (vec1 . vec2) / (||vec1|| * ||vec2||)"""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)


# Compute relevancy score for each document
scores = [cosine_similarity(query_tfidf, doc_vec) for doc_vec in tfidf_matrix]

# Display Results
results = pd.DataFrame({"Document": ["doc1", "doc2", "doc3"], "Score": scores})
results = results.sort_values(by="Score", ascending=False)

print("=== Search Results Ranked by Relevancy ===")
results
# print("=== TF-IDF Matrix (Documents) ===")
# df_tfidf


=== Search Results Ranked by Relevancy ===


,Document,Score
2,doc3,0.27735
0,doc1,0.00000
1,doc2,0.00000
